# Foundation-Sec-8B QLoRA — Kaggle (free T4)

Fine-tunes **fdtn-ai/Foundation-Sec-8B-Instruct** on `data/train.jsonl`, then exports a
merged **GGUF (Q4_K_M)**; `make_modelfile.py` then writes the Ollama
Modelfile on your machine.

**Before running:** enable *GPU T4* and *Internet* in the Kaggle sidebar, and add `data/`
as a Kaggle Dataset input. Run with `SMOKE = True` once to validate the whole path.

In [ ]:
SMOKE = True            # True = 50 rows / 10 steps (validate); False = full run
BASE_MODEL = "fdtn-ai/Foundation-Sec-8B-Instruct"
MAX_SEQ_LEN = 2048
SEED = 42
OUT_GGUF = "/tmp/gguf_model"   # big export files live outside /kaggle/working (~20 GB output limit)

In [ ]:
# Kaggle internet must be ON (sidebar). Unsloth pulls transformers/trl/peft/bitsandbytes.
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

In [ ]:
# Use Foundation-Sec's own chat template: the training data and make_modelfile.py are built for it.
# (Unsloth loads this model as a "legacy tokenizer" and may not pick the template up by itself.)
from huggingface_hub import hf_hub_download
tokenizer.chat_template = open(hf_hub_download("fdtn-ai/Foundation-Sec-8B-Instruct", "chat_template.jinja")).read()
assert tokenizer.chat_template.startswith("{{ bos_token }}"), "unexpected chat template"

In [ ]:
# Get the training data: use an attached prep output if there is one, otherwise build it here.
import glob, os, shutil
os.makedirs("data", exist_ok=True)
names = ("train.jsonl", "eval.jsonl", "system_prompt.txt")
hits = {n: sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True)) for n in names}
if all(hits.values()):
    for n, h in hits.items():
        shutil.copy(h[0], f"data/{n}")
else:
    try:  # gated sources (xlam, primus_instruct) need the HF_TOKEN secret
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login
        login(UserSecretsClient().get_secret("HF_TOKEN"))
    except Exception as e:
        print("No HF_TOKEN secret, gated datasets will be skipped:", e)
    out = os.path.abspath("data")
    if not os.path.isdir("/kaggle/working/cyber-llm"):
        !git clone -q https://github.com/mustafacocgun/cyber-llm.git /kaggle/working/cyber-llm
    !cd /kaggle/working/cyber-llm && git pull -q && python prepare_data.py --out {out}
print(sorted(os.listdir("data")))

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "data/train.jsonl", "eval": "data/eval.jsonl"})

def to_text(ex):
    # Rows are already in Foundation-Sec's own format (system/user/assistant, tool calls
    # as <tool_call> text), so the model's chat template renders them as-is.
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
    return {"text": text.removeprefix(tokenizer.bos_token or "")}  # tokenizer adds BOS itself

ds = ds.map(to_text, remove_columns=ds["train"].column_names)  # keep only "text"
if SMOKE:
    ds["train"] = ds["train"].select(range(min(50, len(ds["train"]))))
print(ds)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    output_dir="out", dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=(2 if SMOKE else 40),
    num_train_epochs=1, max_steps=(10 if SMOKE else -1),
    logging_steps=5, optim="paged_adamw_8bit", seed=SEED, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds["train"], args=args)
trainer.train()

# Safety net: save the LoRA adapter (~300 MB) now, so a failed export never loses the run
model.save_pretrained("/kaggle/working/lora_adapter")
tokenizer.save_pretrained("/kaggle/working/lora_adapter")

In [ ]:
# Quick held-out sanity check (not a full eval — see DATA_PLAN.md).
print(trainer.evaluate(eval_dataset=ds["eval"]))

## Export to GGUF

The trained LoRA adapter is already saved in `/kaggle/working/lora_adapter`, so if this step
fails, the training run is not lost: the adapter can be exported again in a short session.

Unsloth's GGUF export merges the adapter into 16-bit weights, then converts and quantizes
to Q4_K_M (look for "Merging 4bit and LoRA weights" in the log). Those intermediate files
are large (~16 GB each), so they go to `/tmp`; only the final ~5 GB GGUF is copied into
`/kaggle/working` as output.

In [ ]:
!df -h /kaggle/working /tmp
model.save_pretrained_gguf(OUT_GGUF, tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob, os, shutil
ggufs = sorted(glob.glob("/tmp/**/*.gguf", recursive=True))
print("GGUF files:", ggufs)
q4 = [g for g in ggufs if "q4_k_m" in g.lower()] or ggufs
assert q4, "no .gguf produced: check the export log above"
shutil.copy(q4[0], "/kaggle/working/")
print("saved to output:", os.path.basename(q4[0]))
# Download it from the Output panel. Then on your own machine, in the cyber-llm repo:
#   python3 make_modelfile.py ~/Downloads/<file>.gguf
#   ollama create cyber-sec -f ~/Downloads/Modelfile && ollama run cyber-sec

Only the ~5 GB GGUF comes down to your machine. Everything above ran on Kaggle's network.